In [ ]:
import os

In [ ]:
import_gist("8808fb57da9397fc7bab671c4e6241e9")

from sono import *

# HTMX v4 Support for FastHTML

**Objective:** Add htmx v4 compatibility to FastHTML.

**Approach:**
- Create v4-compatible versions of affected functions, using a `4` suffix (e.g., `func4`)
- Add an `htmx4: bool` parameter to `FastHTML.__init__` to toggle between versions
- Default to `False` initially for backward compatibility; switch default in a future release

This project includes a patch package `hx4_patch`. Running `from hx4_patch.core import *` should perform all necessary setup for htmx4.

The source code is in `/app/data/fasthtml-example/hx4_patch/hx4_patch/core.py`

The hx4_patch package is an nbdev project developed in `./hx4_patch` in a notebook, then exported to `.py` using nbdev so we can import it.

When you need detailed notes and examples of patches, read the file `/app/data/fasthtml-example/hx4_patch/hx4_patch/core.py`. It currently contains examples of WebSocket and SSE in htmx4.

## Testing

We test by reimplementing all the examples inside solveit, using htmx v4. To make FastHTML work in solveit, we need to run `from fasthtml.jupyter import *` and `srv = JupyUvi(app)`.

### Automated browser testing

We also test migrated apps using **solveit-chrome** CDP integration via the **solvecdp** Python package (`pip install solvecdp`). The test dialog is at `/fasthtml-example/hx4_patch/nbs/test_htmx4`.

When creating a dialog based on an existing .py example, keep the same code structure but make these changes for htmx v4:
1. Add `from hx4_patch.core import *` after other imports
2. Use `FastHTML` or `fast_app` with `htmx4=True`, `hxmx=False`
3. Replace `serve()` with `JupyUvi(app)`

**Important:** Do NOT re-import `from fasthtml.common import *` or `from fasthtml.jupyter import *` after `from hx4_patch.core import *`. Re-importing overwrites the patched functions (e.g. `def_hdrs`, `FastHTML.__init__`) with the originals, causing errors like `def_hdrs() got multiple values for argument 'surreal'`. The CRAFT already runs the necessary imports — just use `from hx4_patch.core import *` and the symbols are available.

**Creating dialogs:** When using `create_dialog`, paths must be relative to the solveit root — use `/fasthtml-example/...` not `/app/data/fasthtml-example/...`. The solveit root is `/app/data/`, so omit that prefix.

Each folder in `./` contains a fasthtml example (`main.py`) that works with htmx v2. We create a corresponding dialog `.ipynb` with a `4` suffix (e.g., `gol4.ipynb`) to support htmx v4.

# HTMX v4 guide

## Migrate from htmx 2.x to htmx 4.x.
## Quick Start

There are two major behavioral changes between htmx 2.x and 4.x:

* In htmx 2.0 attribute inheritance is *implicit* by default while in 4.0 it is explicity by default
* In htmx 2.0, `400` and `500` response codes are not swapped by default, whereas in htmx 4.0 these requests will be
  swapped

Add these two config lines to restore htmx 2.x behavior:

```html

<script>
    htmx.config.implicitInheritance = true;
    htmx.config.noSwap = [204, 304, '4xx', '5xx'];
</script>
<script src="https://cdn.jsdelivr.net/npm/htmx.org@next/dist/htmx.min.js"></script>
```

[`implicitInheritance`](/reference/config/htmx-config-implicitInheritance) restores htmx 2's implicit attribute
inheritance. [`noSwap`](/reference/config/htmx-config-noSwap) prevents swapping error responses.

Or load the [`htmx-2-compat`](/docs/extensions/htmx-2-compat) extension, which restores implicit inheritance, old event
names, and previous error-swapping defaults:

```html

<script src="/path/to/htmx.js"></script>
<script src="/path/to/ext/htmx-2-compat.js"></script>
```

Most htmx 2 apps should work with either approach. Then migrate incrementally using this guide.

## What Changed

### `fetch()` replaces `XMLHttpRequest`

All requests use the native [`fetch()` API](https://developer.mozilla.org/en-US/docs/Web/API/Fetch_API). This cannot be
reverted.

### Explicit inheritance

Add [`:inherited`](/docs/features/attribute-inheritance) to any attribute that should inherit down the DOM tree.

```html
<!-- htmx 2: implicit inheritance -->
<div hx-confirm="Are you sure?">
    <button hx-delete="/item/1">Delete</button>
</div>

<!-- htmx 4: explicit inheritance -->
<div hx-confirm:inherited="Are you sure?">
    <button hx-delete="/item/1">Delete</button>
</div>
```

Works on any attribute: [`hx-boost`](/reference/attributes/hx-boost)`:inherited`, [
`hx-target`](/reference/attributes/hx-target)`:inherited`, [`hx-confirm`](/reference/attributes/hx-confirm)`:inherited`,
etc.

Use `:append` to add to an inherited value instead of replacing it:

```html

<div hx-include:inherited="#global-fields">
    <!-- appends .extra to the inherited value -->
    <form hx-include:inherited:append=".extra">...</form>
</div>
```

Revert: [`htmx.config.implicitInheritance`](/reference/config/htmx-config-implicitInheritance) `= true`

### Error responses swap

htmx 4 swaps all HTTP responses. Only [`204`](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/204)
and [`304`](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/304) do not swap.

htmx 2 did not swap `4xx` and `5xx` responses. In htmx 4, if your server returns HTML with a `422` or `500`, that HTML
gets swapped into the target. Design your error responses to work as swap content, or use [
`hx-status`](/reference/attributes/hx-status) to control per-code behavior.

Revert: [`htmx.config.noSwap`](/reference/config/htmx-config-noSwap) `= [204, 304, '4xx', '5xx']`

### [`hx-delete`](/reference/attributes/hx-delete) excludes form data

Like [`hx-get`](/reference/attributes/hx-get), [`hx-delete`](/reference/attributes/hx-delete) no longer includes the
enclosing form's inputs.

Fix: add [`hx-include`](/reference/attributes/hx-include)`="closest form"` where needed.

### No history cache

History no longer caches pages in [localStorage](https://developer.mozilla.org/en-US/docs/Web/API/Window/localStorage).
When navigating back, htmx re-fetches the page and swaps it into `<body>`.

Use [`htmx.config.history`](/reference/config/htmx-config-history) `= "reload"` for a full page reload instead. Use
`htmx.config.history = false` to disable.

### OOB swap order

In htmx 2, out-of-band ([`hx-swap-oob`](/reference/attributes/hx-swap-oob)) elements swapped **before** the main
content.

In htmx 4, the main content swaps first. OOB and `hx-partial` elements swap after (in document order).

This matters if an OOB swap creates or modifies DOM that the main swap depends on. If your app relies on that ordering,
restructure so each swap is independent.

### 60-second timeout

htmx 2 had no timeout (`0`). htmx 4 sets [`defaultTimeout`](/reference/config/htmx-config-defaultTimeout) to `60000`.

Revert: `htmx.config.defaultTimeout = 0`

### Extension loading

Include extension scripts directly. No attribute needed:

```html

<script src="/path/to/htmx.js"></script>
<script src="/path/to/ext/sse.js"></script>
```

Restrict which extensions can load:

```html

<meta name="htmx-config" content='{"extensions": "sse, ws"}'>
```

Extension authors use `htmx.registerExtension(name, methodMap)` to register.

See [Extensions documentation](/docs/extensions/using-extensions) for details.

## Renames and Removals

### Rename `hx-disable`

Do this **before** upgrading. The name `hx-disable` has been reassigned:

- In htmx 2, `hx-disable` meant "skip htmx processing on this element"
- In htmx 4, that role is [`hx-ignore`](/reference/attributes/hx-ignore)
- The name `hx-disable` now does what `hx-disabled-elt` used to do (disable form elements during requests)

Rename in this order to avoid conflicts:

1. Rename `hx-disable` to [`hx-ignore`](/reference/attributes/hx-ignore)
2. Rename `hx-disabled-elt` to [`hx-disable`](/reference/attributes/hx-disable)

### Removed attributes

| Removed          | Use instead                                                                                       |
|------------------|---------------------------------------------------------------------------------------------------|
| `hx-vars`        | [`hx-vals`](/reference/attributes/hx-vals) with `js:` prefix                                      |
| `hx-params`      | [`htmx:config:request`](/reference/events/htmx-config-request) event                              |
| `hx-prompt`      | [`hx-confirm`](/reference/attributes/hx-confirm) with `js:` prefix                                |
| `hx-ext`         | [Include extension script directly](/docs/extensions/using-extensions)                            |
| `hx-disinherit`  | Not needed (inheritance is explicit)                                                              |
| `hx-inherit`     | Not needed (inheritance is explicit)                                                              |
| `hx-request`     | [`hx-config`](/reference/attributes/hx-config)                                                    |
| `hx-history`     | Removed (no [localStorage](https://developer.mozilla.org/en-US/docs/Web/API/Window/localStorage)) |
| `hx-history-elt` | Removed                                                                                           |

### Renamed events

All events follow a new pattern: `htmx:phase:action[:sub-action]`

All error events are consolidated to [`htmx:error`](/reference/events/htmx-error).

| htmx 2.x                    | htmx 4.x                                                                          |
|-----------------------------|-----------------------------------------------------------------------------------|
| `htmx:afterOnLoad`          | [`htmx:after:init`](/reference/events/htmx-after-init)                            |
| `htmx:afterProcessNode`     | [`htmx:after:init`](/reference/events/htmx-after-init)                            |
| `htmx:afterRequest`         | [`htmx:after:request`](/reference/events/htmx-after-request)                      |
| `htmx:afterSettle`          | [`htmx:after:swap`](/reference/events/htmx-after-swap)                            |
| `htmx:afterSwap`            | [`htmx:after:swap`](/reference/events/htmx-after-swap)                            |
| `htmx:beforeCleanupElement` | [`htmx:before:cleanup`](/reference/events/htmx-before-cleanup)                    |
| `htmx:beforeHistorySave`    | [`htmx:before:history:update`](/reference/events/htmx-before-history-update)      |
| `htmx:beforeOnLoad`         | [`htmx:before:init`](/reference/events/htmx-before-init)                          |
| `htmx:beforeProcessNode`    | [`htmx:before:process`](/reference/events/htmx-before-process)                    |
| `htmx:beforeRequest`        | [`htmx:before:request`](/reference/events/htmx-before-request)                    |
| `htmx:beforeSwap`           | [`htmx:before:swap`](/reference/events/htmx-before-swap)                          |
| `htmx:configRequest`        | [`htmx:config:request`](/reference/events/htmx-config-request)                    |
| `htmx:historyCacheMiss`     | [`htmx:before:history:restore`](/reference/events/htmx-before-restore-history)    |
| `htmx:historyRestore`       | [`htmx:before:history:restore`](/reference/events/htmx-before-restore-history)    |
| `htmx:load`                 | [`htmx:after:init`](/reference/events/htmx-after-init)                            |
| `htmx:oobAfterSwap`         | [`htmx:after:swap`](/reference/events/htmx-after-swap)                            |
| `htmx:oobBeforeSwap`        | [`htmx:before:swap`](/reference/events/htmx-before-swap)                          |
| `htmx:pushedIntoHistory`    | [`htmx:after:history:push`](/reference/events/htmx-after-push-into-history)       |
| `htmx:replacedInHistory`    | [`htmx:after:history:replace`](/reference/events/htmx-after-replace-into-history) |
| `htmx:responseError`        | [`htmx:error`](/reference/events/htmx-error)                                      |
| `htmx:sendError`            | [`htmx:error`](/reference/events/htmx-error)                                      |
| `htmx:swapError`            | [`htmx:error`](/reference/events/htmx-error)                                      |
| `htmx:targetError`          | [`htmx:error`](/reference/events/htmx-error)                                      |
| `htmx:timeout`              | [`htmx:error`](/reference/events/htmx-error)                                      |

### `hx-on::` shorthand

The double-colon shorthand for htmx events no longer works. Because event names changed from camelCase to
colon-separated (e.g. `htmx:afterRequest` → `htmx:after:request`), the `hx-on::` prefix can no longer map to the correct
event name.

Use the full event name instead:

```html
<!-- htmx 2 -->
<form hx-on::after-request="this.reset()">

    <!-- htmx 4 -->
    <form hx-on:htmx:after:request="this.reset()">
```

This applies to all `hx-on::` event handlers. Find and replace `hx-on::` with `hx-on:htmx:` and update the event name to
the new colon-separated format (see table above).

### Removed events

Validation events are removed. Use native browser form validation:

- `htmx:validation:validate`
- `htmx:validation:failed`
- `htmx:validation:halted`

XHR events are removed (htmx uses `fetch()` now):

| Removed              | Use instead                                                      |
|----------------------|------------------------------------------------------------------|
| `htmx:xhr:loadstart` | No replacement                                                   |
| `htmx:xhr:loadend`   | [`htmx:finally:request`](/reference/events/htmx-finally-request) |
| `htmx:xhr:progress`  | No replacement                                                   |
| `htmx:xhr:abort`     | [`htmx:error`](/reference/events/htmx-error)                     |

### Config changes

**Renamed:**

| htmx 2.x                 | htmx 4.x                                                                   |
|--------------------------|----------------------------------------------------------------------------|
| `defaultSwapStyle`       | [`defaultSwap`](/reference/config/htmx-config-defaultSwap)                 |
| `globalViewTransitions`  | [`transitions`](/reference/config/htmx-config-transitions)                 |
| `historyEnabled`         | [`history`](/reference/config/htmx-config-history)                         |
| `includeIndicatorStyles` | [`includeIndicatorCSS`](/reference/config/htmx-config-includeIndicatorCSS) |
| `timeout`                | [`defaultTimeout`](/reference/config/htmx-config-defaultTimeout)           |

**Changed defaults:**

| Config                                                                   | htmx 2           | htmx 4               |
|--------------------------------------------------------------------------|------------------|----------------------|
| [`defaultTimeout`](/reference/config/htmx-config-defaultTimeout)         | `0` (no timeout) | `60000` (60 seconds) |
| [`defaultSettleDelay`](/reference/config/htmx-config-defaultSettleDelay) | `20`             | `1`                  |

**Removed:**

`addedClass`, `allowEval`, `allowNestedOobSwaps`, `allowScriptTags`, `attributesToSettle`, `defaultSwapDelay`,
`disableSelector` (use [`hx-ignore`](/reference/attributes/hx-ignore)), `getCacheBusterParam`, `historyCacheSize`,
`ignoreTitle` (still works per-swap via [`hx-swap`](/reference/attributes/hx-swap)`="... ignoreTitle:true"`),
`methodsThatUseUrlParams`, `refreshOnHistoryMiss`, `responseHandling` (use [
`hx-status`](/reference/attributes/hx-status) and [`noSwap`](/reference/config/htmx-config-noSwap)), `scrollBehavior`,
`scrollIntoViewOnBoost`, `selfRequestsOnly` (use [`htmx.config.mode`](/reference/config/htmx-config-mode)),
`settlingClass`, `swappingClass`, `triggerSpecsCache`, `useTemplateFragments`, `withCredentials` (use [
`hx-config`](/reference/attributes/hx-config)), `wsBinaryType`, `wsReconnectDelay`

The `htmx-swapping`, `htmx-settling`, and `htmx-added` CSS classes are still applied during swaps. The config keys to
customize their names have been removed.

### Request headers

| htmx 2.x          | htmx 4.x                                                | Notes                                                                  |
|-------------------|---------------------------------------------------------|------------------------------------------------------------------------|
| `HX-Trigger`      | [`HX-Source`](/reference/headers/HX-Source)             | Format changed to `tagName#id` (e.g. `button#submit`)                  |
| `HX-Target`       | [`HX-Target`](/reference/headers/HX-Target)             | Format changed to `tagName#id`                                         |
| `HX-Trigger-Name` | removed                                                 | Use [`HX-Source`](/reference/headers/HX-Source)                        |
| `HX-Prompt`       | removed                                                 | Use [`hx-confirm`](/reference/attributes/hx-confirm) with `js:` prefix |
| *(new)*           | [`HX-Request-Type`](/reference/headers/HX-Request-Type) | `"full"` or `"partial"`                                                |
| *(new)*           | [`Accept`](/reference/headers/Accept)                   | Now explicitly `text/html`                                             |

### Response headers

Removed:

- `HX-Trigger-After-Swap`
- `HX-Trigger-After-Settle`

Use [`HX-Trigger`](/reference/headers/HX-Trigger) or JavaScript instead.

Unchanged: [`HX-Trigger`](/reference/headers/HX-Trigger), [`HX-Location`](/reference/headers/HX-Location), [
`HX-Push-Url`](/reference/headers/HX-Push-Url), [`HX-Redirect`](/reference/headers/HX-Redirect), [
`HX-Refresh`](/reference/headers/HX-Refresh), [`HX-Replace-Url`](/reference/headers/HX-Replace-Url), `HX-Retarget`,
`HX-Reswap`, `HX-Reselect`.

### JavaScript API changes

**Removed methods.** Use native JavaScript:

| htmx 2.x             | Use instead                                                            |
|----------------------|------------------------------------------------------------------------|
| `htmx.addClass()`    | `element.classList.add()`                                              |
| `htmx.removeClass()` | `element.classList.remove()`                                           |
| `htmx.toggleClass()` | `element.classList.toggle()`                                           |
| `htmx.closest()`     | `element.closest()`                                                    |
| `htmx.remove()`      | `element.remove()`                                                     |
| `htmx.off()`         | `removeEventListener()` (`htmx.on()` returns the callback)             |
| `htmx.location()`    | `htmx.ajax()`                                                          |
| `htmx.logAll()`      | [`htmx.config.logAll`](/reference/config/htmx-config-logAll) `= true`  |
| `htmx.logNone()`     | [`htmx.config.logAll`](/reference/config/htmx-config-logAll) `= false` |

**Renamed:** `htmx.defineExtension()` is now `htmx.registerExtension()`.

**Still available:** `htmx.ajax()`, `htmx.config`, `htmx.find()`, `htmx.findAll()`, `htmx.on()`, `htmx.onLoad()`,
`htmx.parseInterval()`, `htmx.process()`, `htmx.swap()`, `htmx.trigger()`.

Note: `htmx.onLoad()` now listens on [`htmx:after:process`](/reference/events/htmx-after-process), not [
`htmx:after:init`](/reference/events/htmx-after-init).

## What's New

### Attributes

| Attribute                                          | Purpose                                                               |
|----------------------------------------------------|-----------------------------------------------------------------------|
| [`hx-action`](/reference/attributes/hx-action)     | Specify URL (use with [`hx-method`](/reference/attributes/hx-method)) |
| [`hx-method`](/reference/attributes/hx-method)     | Specify HTTP method                                                   |
| [`hx-config`](/reference/attributes/hx-config)     | Per-element request config (JSON or `key:value` syntax)               |
| [`hx-ignore`](/reference/attributes/hx-ignore)     | Disable htmx processing (was `hx-disable`)                            |
| [`hx-validate`](/reference/attributes/hx-validate) | Control form validation behavior                                      |

### [`hx-swap`](/reference/attributes/hx-swap) styles

```html

<div hx-get="/data" hx-swap="innerMorph">...</div>
<div hx-get="/data" hx-swap="outerMorph">...</div>
<div hx-get="/text" hx-swap="textContent">...</div>
<div hx-get="/remove" hx-swap="delete">...</div>
```

- `innerMorph` / `outerMorph`: morph swaps using the idiomorph algorithm. Better for preserving state in complex UIs.
- `textContent`: set the target's text content (no HTML parsing).
- `delete`: remove the target element entirely.

New aliases for existing swap styles (both old and new names work):

| New       | Equivalent to |
|-----------|---------------|
| `before`  | `beforebegin` |
| `after`   | `afterend`    |
| `prepend` | `afterbegin`  |
| `append`  | `beforeend`   |

### [Status code swaps](/reference/attributes/hx-status)

Set different swap behavior per HTTP status code:

```html

<form hx-post="/save"
      hx-status:422="swap:innerHTML target:#errors select:#validation-errors"
      hx-status:5xx="swap:none push:false">
    <!-- form fields -->
</form>
```

Available config keys: `swap:`, `target:`, `select:`, `push:`, `replace:`, `transition:`.

Supports exact codes (`404`), single-digit wildcards (`50x`), and range wildcards (`5xx`). Evaluated in order of
specificity.

### `<hx-partial>`

Target multiple elements from one response:

```html

<hx-partial hx-target="#messages" hx-swap="beforeend">
    <div>New message</div>
</hx-partial>

<hx-partial hx-target="#count">
    <span>5</span>
</hx-partial>
```

Each `<hx-partial>` specifies its own [`hx-target`](/reference/attributes/hx-target) and [
`hx-swap`](/reference/attributes/hx-swap) strategy. A cleaner alternative to out-of-band swaps.

### Etag support

htmx 4 supports Etag-based conditional requests automatically:

- Response includes an [`Etag`](/reference/headers/ETag) header: htmx stores it on the source element
- Next request from that element includes an [`If-None-Match`](/reference/headers/If-None-Match) header
- `304 Not Modified` responses do not swap, avoiding unnecessary DOM updates

### View transitions

[View Transitions API](https://developer.mozilla.org/en-US/docs/Web/API/View_Transition_API) support is available but
disabled by default.

Enable: [`htmx.config.transitions`](/reference/config/htmx-config-transitions) `= true`

### JSX compatibility

Frameworks that don't support `:` in attribute names can use [
`metaCharacter`](/reference/config/htmx-config-metaCharacter) to replace it:

```js
htmx.config.metaCharacter = "-";
// hx-ws-connect instead of hx-ws:connect
// hx-confirm-inherited instead of hx-confirm:inherited
```

### JavaScript methods

- `htmx.forEvent(eventName, timeout)`: returns a promise that resolves when an event fires
- `htmx.takeClass(element, className, container)`: removes class from siblings, adds to element
- `htmx.timeout(time)`: returns a promise that resolves after a delay

### Request context

All events provide a consistent `ctx` object with request/response information.

### Events

| Event                                                                        | Fires                                             |
|------------------------------------------------------------------------------|---------------------------------------------------|
| [`htmx:after:cleanup`](/reference/events/htmx-after-cleanup)                 | After element cleanup                             |
| [`htmx:after:history:update`](/reference/events/htmx-after-history-update)   | After history update                              |
| [`htmx:after:process`](/reference/events/htmx-after-process)                 | After element processing                          |
| [`htmx:before:response`](/reference/events/htmx-before-response)             | Before response body is read (cancellable)        |
| [`htmx:before:settle`](/reference/events/htmx-before-settle)                 | Before settle phase                               |
| [`htmx:after:settle`](/reference/events/htmx-after-settle)                   | After settle phase                                |
| [`htmx:before:viewTransition`](/reference/events/htmx-before-viewTransition) | Before a view transition starts (cancellable)     |
| [`htmx:after:viewTransition`](/reference/events/htmx-after-viewTransition)   | After a view transition completes                 |
| [`htmx:finally:request`](/reference/events/htmx-finally-request)             | Always fires after a request (success or failure) |

### Config keys

| Config                                                                 | Default         | Purpose                                                       |
|------------------------------------------------------------------------|-----------------|---------------------------------------------------------------|
| [`extensions`](/reference/config/htmx-config-extensions)               | `''`            | Comma-separated list of allowed extension names               |
| [`mode`](/reference/config/htmx-config-mode)                           | `'same-origin'` | Fetch mode (replaces `selfRequestsOnly`)                      |
| [`inlineScriptNonce`](/reference/config/htmx-config-inlineScriptNonce) | `''`            | Nonce for inline scripts                                      |
| [`inlineStyleNonce`](/reference/config/htmx-config-inlineStyleNonce)   | `''`            | Nonce for inline styles                                       |
| [`metaCharacter`](/reference/config/htmx-config-metaCharacter)         | `':'`           | Separator character in attribute/event names                  |
| [`morphIgnore`](/reference/config/htmx-config-morphIgnore)             | `''`            | CSS selector for elements to ignore during morph              |
| [`morphScanLimit`](/reference/config/htmx-config-morphScanLimit)       |                 | Max elements to scan during morph matching                    |
| [`morphSkip`](/reference/config/htmx-config-morphSkip)                 | `''`            | CSS selector for elements to skip during morph                |
| [`morphSkipChildren`](/reference/config/htmx-config-morphSkipChildren) | `''`            | CSS selector for elements whose children to skip during morph |

### SSE extension

The SSE extension uses `fetch()` and [`ReadableStream`](https://developer.mozilla.org/en-US/docs/Web/API/ReadableStream)
instead of [`EventSource`](https://developer.mozilla.org/en-US/docs/Web/API/EventSource). This enables request bodies,
custom headers, and all HTTP methods.

See the [SSE extension documentation](/docs/extensions/sse) for details.

### Core extensions

htmx 4 ships with 9 core extensions:

| Extension                                                 | Description                                                                          |
|-----------------------------------------------------------|--------------------------------------------------------------------------------------|
| [`alpine-compat`](/docs/extensions/alpine-compat)         | Alpine.js compatibility: initializes Alpine on fragments before swap                 |
| [`browser-indicator`](/docs/extensions/browser-indicator) | Shows the browser's native loading indicator during requests                         |
| [`head-support`](/docs/extensions/head-support)           | Merges head tag information (styles, etc.) in htmx requests                          |
| [`htmx-2-compat`](/docs/extensions/htmx-2-compat)         | Restores implicit inheritance, old event names, and previous error-swapping defaults |
| [`optimistic`](/docs/extensions/optimistic)               | Shows expected content from a template before the server responds                    |
| [`preload`](/docs/extensions/preload)                     | Triggers requests early (on mouseover/mousedown) for near-instant page loads         |
| [`sse`](/docs/extensions/sse)                             | Server-Sent Events streaming support                                                 |
| [`upsert`](/docs/extensions/upsert)                       | Updates existing elements by ID and inserts new ones, preserving unmatched elements  |
| [`ws`](/docs/extensions/ws)                               | Bi-directional Web Socket communication                                              |

## Checklist

1. Add config options or load [`htmx-2-compat`](/docs/extensions/htmx-2-compat) for backward compatibility
2. Rename `hx-disable` to [`hx-ignore`](/reference/attributes/hx-ignore), then `hx-disabled-elt` to [
   `hx-disable`](/reference/attributes/hx-disable)
3. Replace removed attributes with alternatives
4. Find/replace event names in JavaScript and `hx-on::` attributes
5. Replace removed API methods with native JS
6. Update extensions
7. Rename changed config keys
8. Test error handling (4xx/5xx now swap by default)
9. Test attribute inheritance
10. Test history navigation

## Migration Notes

Individual documentation pages include migration notes where features changed.

Look for these:

<details class="warning">
<summary>Changes in htmx 4.0</summary>

</details>

## Get Help

- [GitHub Discussions](https://github.com/bigskysoftware/htmx/discussions)
- [Discord](https://htmx.org/discord)
- [Examples](/examples)

# WebSocket

---
title: "WebSockets"
description: "Enable bidirectional real-time communication via WebSockets"
keywords: ["websockets", "ws", "real-time", "bidirectional", "socket"]
---

The WebSocket extension enables real-time, bidirectional communication with [WebSocket](https://developer.mozilla.org/en-US/docs/Web/API/WebSockets_API/Writing_WebSocket_client_applications) servers directly from HTML. It manages connections efficiently through reference counting, automatic reconnection, and seamless integration with htmx's swap and event model.

## Installing

```html
<script src="/path/to/htmx.js"></script>
<script src="/path/to/ext/hx-ws.js"></script>
```

For npm-style build systems:

```javascript
import 'htmx.org';
import 'htmx.org/dist/ext/hx-ws.js';
```

## Usage

| Attribute | Description |
|-----------|-------------|
| `hx-ws:connect="<url>"` | Establishes a WebSocket connection to the specified URL |
| `hx-ws:send` | Sends form data or [`hx-vals`](/reference/attributes/hx-vals) to the WebSocket on trigger |
| `hx-ws:send="<url>"` | Like `hx-ws:send` but creates its own connection to the URL |

**JSX-Compatible Variants:** For frameworks that don't support colons in attribute names, use hyphen variants: `hx-ws-connect` and `hx-ws-send`.

### Basic Example

```html
<div hx-ws:connect="/chatroom" hx-target="#messages" hx-swap="beforeend">
    <div id="messages"></div>
    <form hx-ws:send>
        <input name="message" placeholder="Type a message...">
        <button type="submit">Send</button>
    </form>
</div>
```

This example:
1. Establishes a WebSocket connection to `/chatroom` when the page loads
2. Appends incoming HTML messages to `#messages`
3. Sends form data as JSON when the form is submitted

## Receiving Messages

### JSON Envelope Format

Messages from the server should be JSON objects:

```json
{
    "channel": "ui",
    "format": "html",
    "target": "#notifications",
    "swap": "beforeend",
    "payload": "<div class='notification'>New message!</div>",
    "request_id": "abc-123"
}
```

| Field | Default | Description |
|-------|---------|-------------|
| `channel` | `"ui"` | Message routing channel |
| `format` | `"html"` | Content format |
| `target` | Element's [`hx-target`](/reference/attributes/hx-target) | CSS selector for target element |
| `swap` | Element's [`hx-swap`](/reference/attributes/hx-swap) | Swap strategy (innerHTML, beforeend, etc.) |
| `payload` | — | The content to swap |
| `request_id` | — | Matches response to original request |

**Minimal Example** (using all defaults):

```json
{"payload": "<div>Hello World</div>"}
```

### Channels

- **`ui` channel** (default): HTML content is swapped into the target element using htmx's swap pipeline
- **Custom channels**: Emit an `htmx:wsMessage` event for application handling

```javascript
document.addEventListener('htmx:wsMessage', (e) => {
    if (e.detail.channel === 'notifications') {
        showNotification(e.detail.payload);
    }
});
```

## Sending Messages

When an element with `hx-ws:send` is triggered, the extension sends a JSON message:

```json
{
    "type": "request",
    "request_id": "550e8400-e29b-41d4-a716-446655440000",
    "event": "submit",
    "headers": {
        "HX-Request": "true",
        "HX-Current-URL": "https://example.com/chat"
    },
    "values": {
        "message": "Hello!"
    },
    "path": "wss://example.com/chatroom",
    "id": "chat-form"
}
```

### Modifying Messages Before Send

```javascript
document.addEventListener('htmx:before:ws:send', (e) => {
    e.detail.data.headers['Authorization'] = 'Bearer ' + getToken();

    if (!isValid(e.detail.data)) {
        e.preventDefault();
    }
});
```

## Configuration

Configure the extension via `htmx.config.websockets`:

```javascript
htmx.config.websockets = {
    reconnect: true,           // Enable auto-reconnect (default: true)
    reconnectDelay: 1000,      // Initial reconnect delay in ms (default: 1000)
    reconnectMaxDelay: 30000,  // Maximum reconnect delay in ms (default: 30000)
    reconnectJitter: true,     // Add randomization to delays (default: true)
    pendingRequestTTL: 30000   // Time-to-live for pending requests in ms (default: 30000)
};
```

### Reconnection Strategy

The extension uses exponential backoff with optional jitter:

- **Base formula**: `delay = min(reconnectDelay * 2^(attempts-1), reconnectMaxDelay)`
- **Jitter**: Adds +/-25% randomization to avoid thundering herd
- **Reset**: Attempts counter resets to 0 on successful connection

## Connection Management

### Reference Counting

Multiple elements can share a single WebSocket connection:

```html
<div hx-ws:connect="/notifications" id="notif-1"></div>
<div hx-ws:connect="/notifications" id="notif-2"></div>
```

When all elements using a connection are removed from the DOM, the connection is automatically closed.

## Events

### Connection Lifecycle

| Event | Cancelable | Detail | Description |
|-------|------------|--------|-------------|
| `htmx:before:ws:connect` | Yes | `{url}` | Before establishing connection |
| `htmx:after:ws:connect` | No | `{url, socket}` | After successful connection |
| `htmx:ws:close` | No | `{url, code, reason}` | When connection closes |
| `htmx:ws:error` | No | `{url, error}` | On connection error |
| `htmx:ws:reconnect` | No | `{url, attempts}` | Before reconnection attempt |

### Message Events

| Event | Cancelable | Detail | Description |
|-------|------------|--------|-------------|
| `htmx:before:ws:send` | Yes | `{data, element, url}` | Before sending (data is modifiable) |
| `htmx:after:ws:send` | No | `{data, url}` | After message sent |
| `htmx:before:ws:message` | Yes | `{envelope, element}` | Before processing received message |
| `htmx:after:ws:message` | No | `{envelope, element}` | After processing received message |
| `htmx:wsMessage` | No | `{channel, format, payload, ...}` | For non-UI channel messages |

## Examples

### Live Chat

```html
<div hx-ws:connect="/chat">
    <div id="messages" hx-target="this" hx-swap="beforeend"></div>
    <form hx-ws:send>
        <input name="message" placeholder="Message..." autocomplete="off">
        <button type="submit">Send</button>
    </form>
</div>
```

### Real-Time Dashboard

```html
<div hx-ws:connect="/dashboard">
    <div id="cpu-usage">--</div>
    <div id="memory-usage">--</div>
</div>
```

Server sends targeted updates:

```json
{"target": "#cpu-usage", "payload": "<span>45%</span>"}
{"target": "#memory-usage", "payload": "<span>2.3 GB</span>"}
```

## Upgrading from htmx 2.x

### Attribute Changes

| Old (htmx 2.x) | New (htmx 4.x) | Notes |
|----------------|----------------|-------|
| `ws-connect="<url>"` | `hx-ws:connect="<url>"` | Or `hx-ws-connect` for JSX |
| `ws-send` | `hx-ws:send` | Or `hx-ws-send` for JSX |

### Event Changes

| Old Event | New Event | Notes |
|-----------|-----------|-------|
| `htmx:wsOpen` | `htmx:after:ws:connect` | Different detail structure |
| `htmx:wsClose` | `htmx:ws:close` | Now includes `code` and `reason` |
| `htmx:wsError` | `htmx:ws:error` | Similar |
| `htmx:wsBeforeMessage` | `htmx:before:ws:message` | Different detail structure |
| `htmx:wsAfterMessage` | `htmx:after:ws:message` | Different detail structure |
| `htmx:wsConfigSend` | `htmx:before:ws:send` | Modify `e.detail.data` instead |
| `htmx:wsAfterSend` | `htmx:after:ws:send` | Similar |

### Message Format Changes

**Send payload** now includes `type`, `request_id`, `event`, and structured `headers` object instead of `HEADERS` string.

**Receive format** now expects JSON envelope with `channel`, `format`, `target`, `swap`, `payload` fields instead of raw HTML or [`hx-swap-oob`](/reference/attributes/hx-swap-oob).

## FastHTML WS approach for htmx v4

Although htmx v4 WebSockets support a JSON envelope format for server→client messages, **we send raw HTML elements** (same as htmx v2) for compatibility. This means the same `_send_ws` function works for both v2 and v4 — no separate send function is needed. The only patch required is `_find_wsp` to handle htmx v4's form data nesting (`data['values']` instead of top-level).

# SSE

## SSE htmx v4 doc
---
title: "Server-Sent Events (SSE)"
description: "Stream server updates using Server-Sent Events"
keywords: ["sse", "server-sent events", "event stream", "streaming", "real-time"]
---

The SSE extension adds support for [Server-Sent Events](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events) streaming to htmx. It works by intercepting any htmx response with `Content-Type: text/event-stream` and streaming SSE messages into the DOM in real-time.

SSE is a lightweight alternative to WebSockets that works over existing HTTP connections, making it easy to use through proxy servers and firewalls. SSE is uni-directional: the server pushes data to the client. If you need bi-directional communication, consider [WebSockets](/docs/extensions/ws) instead.

## Installing

Include the extension script after htmx:

```html
<script src="/path/to/htmx.js"></script>
<script src="/path/to/ext/hx-sse.js"></script>
```

## How It Works

The SSE extension hooks into htmx's request pipeline. When any htmx request receives a response with `Content-Type: text/event-stream`, the extension takes over and streams SSE messages into the DOM instead of performing a normal swap.

This means **any [`hx-get`](/reference/attributes/hx-get), [`hx-post`](/reference/attributes/hx-post), etc. that returns an SSE stream will just work**, no special attributes needed beyond loading the extension.

## `hx-sse:connect`

For persistent SSE connections (auto-connect on load, reconnect on failure), use `hx-sse:connect`:

```html
<!-- Auto-connects on load, streams messages into the div -->
<div hx-sse:connect="/stream">
    Waiting for messages...
</div>
```

`hx-sse:connect` is convenience sugar for a well-preconfigured `hx-get`. It defaults to:
- **Trigger**: `load` (connects immediately)
- **Reconnect**: enabled with exponential backoff
- **Pause on background**: closes the stream when the tab is backgrounded, reconnects when visible

### Using with Standard Attributes

`hx-sse:connect` works with all standard htmx attributes:

```html
<!-- Swap into a different target -->
<button hx-sse:connect="/notifications" hx-target="#alerts">
    Start Notifications
</button>
<div id="alerts"></div>

<!-- Append messages instead of replacing -->
<div hx-sse:connect="/log" hx-swap="beforeend">
    <h3>Log:</h3>
</div>
```

### Trigger Modifiers

All standard [`hx-trigger`](/reference/attributes/hx-trigger) modifiers are supported:

```html
<!-- Connect after a delay -->
<div hx-sse:connect="/stream" hx-trigger="load delay:2s">

<!-- Connect on click -->
<button hx-sse:connect="/stream" hx-trigger="click">Start</button>

<!-- Connect on click, only once -->
<button hx-sse:connect="/stream" hx-trigger="click once">Start</button>
```

### Using Standard htmx Attributes

Since the extension intercepts based on Content-Type, any htmx request that returns `text/event-stream` will be streamed automatically:

```html
<!-- hx-get, hx-post, etc. all work -->
<div hx-get="/stream" hx-trigger="load">
    Waiting...
</div>

<button hx-post="/generate" hx-target="#output">
    Generate
</button>
```

The difference is that `hx-sse:connect` enables reconnection and `pauseOnBackground` by default, while standard attributes do not.

## `hx-sse:close`

Use `hx-sse:close` to gracefully close an SSE connection when a specific named event is received from the server:

```html
<div hx-sse:connect="/stream" hx-sse:close="done">
    Streaming until server sends "done"...
</div>
```

When the server sends `event: done`, the connection is closed and an `htmx:sse:close` event is fired with `detail.reason === "message"`.

## Named Events

SSE messages with an `event:` field are dispatched as DOM events on the source element rather than being swapped:

```txt
event: notification
data: {"title": "New message", "body": "Hello!"}
```

```html
<div hx-sse:connect="/events"
     hx-on:notification="alert(event.detail.data)">
</div>
```

Messages without an `event:` field are swapped into the DOM as HTML content.

## Configuration

Configure SSE behavior globally via `htmx.config.sse` or per-element via [`hx-config`](/reference/attributes/hx-config):

```html
<!-- Global config -->
<meta name="htmx-config" content='{
    "sse": {
        "reconnect": true,
        "reconnectDelay": 500,
        "reconnectMaxDelay": 60000,
        "reconnectMaxAttempts": 50,
        "reconnectJitter": 0.3,
        "pauseOnBackground": false
    }
}'>

<!-- Per-element override -->
<div hx-sse:connect="/stream" hx-config='{"sse": {"reconnect": false}}'>
```

| Option | Default (`hx-sse:connect`) | Default (`hx-get`) | Description |
|--------|---------------------------|---------------------|-------------|
| `reconnect` | `true` | `false` | Auto-reconnect on stream end |
| `reconnectDelay` | `500` | `500` | Initial reconnect delay (ms) |
| `reconnectMaxDelay` | `60000` | `60000` | Maximum reconnect delay (ms) |
| `reconnectMaxAttempts` | `Infinity` | `Infinity` | Maximum reconnection attempts |
| `reconnectJitter` | `0.3` | `0.3` | Jitter factor (0-1) for delay randomization |
| `pauseOnBackground` | `true` | `false` | Disconnect when the tab is backgrounded, reconnect when visible (see [Background Tab Behavior](#background-tab-behavior)) |

### Reconnection Strategy

The extension uses exponential backoff with jitter:

- **Formula**: `delay = min(reconnectDelay × 2^(attempt-1), reconnectMaxDelay)`
- **Jitter**: Adds ±`reconnectJitter` randomization to avoid thundering herd
- **Last-Event-ID**: Automatically sent on reconnection if the server provided message IDs (see [Background Tab Behavior](#background-tab-behavior))

### Background Tab Behavior

When `pauseOnBackground` is enabled (the default for `hx-sse:connect`), the extension disconnects the
stream when the browser tab is hidden and reconnects when the tab becomes visible again. This exists
because some browsers (notably iOS Safari) silently kill SSE connections when the app is backgrounded
without firing any error events, leaving the connection in a zombie state.

**Messages sent by the server while the tab is in the background are not received by the client.** Whether
those messages can be recovered depends on your server:

- If the server includes `id:` fields in its SSE messages, the extension tracks the last received ID and
  sends it as a `Last-Event-ID` header when reconnecting.
- If the server reads the `Last-Event-ID` header and replays missed messages, nothing is lost.
- If the server does not send `id:` fields or does not support `Last-Event-ID`, messages sent during the
  background period are lost.

#### Example: Resumable Notifications Stream

**Server** (Python with FastAPI + sse-starlette):

```python
from fastapi import FastAPI, Request
from sse_starlette.sse import EventSourceResponse

app = FastAPI()
notifications = []  # In production, use a database

@app.get("/notifications")
async def sse(request: Request):
    last_id = request.headers.get("last-event-id")

    async def stream():
        # Replay any missed messages
        start = 0
        if last_id:
            for i, n in enumerate(notifications):
                if str(n["id"]) == last_id:
                    start = i + 1
                    break
            for n in notifications[start:]:
                yield {"id": str(n["id"]), "data": n["data"]}

        # Stream new messages as they arrive
        seen = len(notifications)
        while True:
            if len(notifications) > seen:
                for n in notifications[seen:]:
                    yield {"id": str(n["id"]), "data": n["data"]}
                seen = len(notifications)
            await asyncio.sleep(0.5)

    return EventSourceResponse(stream())
```

**Client:**

```html
<div hx-sse:connect="/notifications" hx-swap="beforeend">
    <!-- Notifications appear here -->
</div>
```

When the user switches tabs and comes back, the extension reconnects with
`Last-Event-ID: <last-received-id>`, and the server replays any notifications
that were sent in the meantime.

## Events

### `htmx:before:sse:connection`

Fired before a connection attempt (initial or reconnection). Set `detail.connection.cancelled = true` to prevent the connection.

```javascript
document.body.addEventListener('htmx:before:sse:connection', function(evt) {
    if (evt.detail.connection.attempt > 10) {
        evt.detail.connection.cancelled = true;
    }
});
```

- `detail.connection.attempt` - attempt number (`0` = initial, `> 0` = reconnection)
- `detail.connection.delay` - the delay before connection (ms), modifiable
- `detail.connection.url` - the SSE endpoint URL
- `detail.connection.lastEventId` - the last event ID received
- `detail.connection.cancelled` - set to `true` to cancel

### `htmx:after:sse:connection`

Fired after a successful connection (or reconnection) to the SSE stream.

- `detail.connection.attempt` - attempt number (`0` = initial, `> 0` = reconnection)
- `detail.connection.url` - the SSE endpoint URL
- `detail.connection.status` - the HTTP status code
- `detail.connection.lastEventId` - the last event ID received

### `htmx:before:sse:message`

Fired before each SSE message is processed. All fields are modifiable.

```javascript
document.body.addEventListener('htmx:before:sse:message', function(evt) {
    // Skip heartbeats
    if (evt.detail.message.event === 'heartbeat') {
        evt.detail.message.cancelled = true;
    }

    // Transform data before swap
    evt.detail.message.data = sanitize(evt.detail.message.data);
});
```

- `detail.message.data` - the message data (modifiable)
- `detail.message.event` - the event type (modifiable)
- `detail.message.id` - the message ID (if specified)
- `detail.message.cancelled` - set to `true` to skip

### `htmx:after:sse:message`

Fired after an SSE message has been processed.

- `detail.message` - same shape as `htmx:before:sse:message`

### `htmx:sse:error`

Fired when a stream error occurs.

- `detail.error` - the error object

### `htmx:sse:close`

Fired when an SSE connection is closed.

- `detail.reason` - why the connection was closed:
  - `"message"` - closed by `hx-sse:close` matching a named event
  - `"removed"` - the element was removed from the DOM
  - `"ended"` - the stream ended naturally or reconnection was exhausted
  - `"cancelled"` - the initial connection was cancelled via `htmx:before:sse:connection`
  - `"cleanup"` - closed during element cleanup (e.g., parent swap)

## Upgrading from htmx 2.x

The htmx 2.x SSE extension (`htmx-ext-sse`) has been rewritten for htmx 4.

### What Changed

The 2.x extension was built around `EventSource` and had its own swap mechanism (`sse-swap`) that operated outside of htmx's normal request/response pipeline. The 4.x extension removes all of that. It hooks into htmx's standard request pipeline instead: any htmx request that receives a `Content-Type: text/event-stream` response is automatically streamed as SSE.

This means:
- **Unnamed messages** (no `event:` field) are swapped into the DOM using htmx's normal swap pipeline.
- **Named messages** (with an `event:` field) are dispatched as DOM events on the source element. They are not swapped.
- `sse-swap` is gone entirely. There is no equivalent, because the extension no longer has its own swap system.

### Connecting and Swapping

**htmx 2.x:**
```html
<div sse-connect="/chatroom" sse-swap="message">
    Contents of this box will be updated in real time
    with every SSE message received from the chatroom.
</div>
```

**htmx 4.x:**
```html
<div hx-sse:connect="/chatroom">
    Contents of this box will be updated in real time
    with every SSE message received from the chatroom.
</div>
```

### Event Changes

| htmx 2.x Event | htmx 4.x Event | Notes |
|-----------------|-----------------|-------|
| `htmx:sseOpen` | `htmx:after:sse:connection` | `detail.connection.attempt === 0` for initial |
| `htmx:sseError` | `htmx:sse:error` | `detail.error` contains the error |
| `htmx:sseBeforeMessage` | `htmx:before:sse:message` | Set `detail.message.cancelled = true` to skip |
| `htmx:sseMessage` | `htmx:after:sse:message` | |
| `htmx:sseClose` | `htmx:sse:close` | `detail.reason` indicates why |

### Other Changes

- **No more `EventSource`**: uses `fetch()` + `ReadableStream`, enabling POST requests, custom headers, and cookies.
- **Reconnection**: `hx-sse:connect` reconnects automatically with exponential backoff. Configure via `hx-config`.
- **Background tab handling**: pauses streams when the tab is backgrounded, reconnects when visible (configurable via `pauseOnBackground`).
- **Any HTTP method**: `hx-post`, [`hx-put`](/reference/attributes/hx-put), etc. all work with SSE responses.

## SSE htmx v2 doc

+++
title = "htmx Server Sent Event (SSE) Extension"
+++

The `Server Sent Events` extension connects to
an [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events) directly
from HTML. It manages the connections to your web server, listens for server events, and then swaps their contents into
your htmx webpage in real-time.

SSE is a lightweight alternative to WebSockets that works over existing HTTP connections, so it is easy to use through
proxy servers and firewalls. Remember, SSE is a uni-directional service, so you cannot send any messages to an SSE
server once the connection has been established. If you need bi-directional communication, then you should consider
using [WebSockets](@/extensions/ws.md) instead.

This extension replaces the experimental `hx-sse` attribute built into previous versions of htmx. For help migrating
from older versions, see the migration guide at the bottom of this page.

Use the following attributes to configure how SSE connections behave:

* `sse-connect="<url>"` - The URL of the SSE server.
* `sse-swap="<message-name>"` - The name of the message to swap into the DOM.
* `hx-trigger="sse:<message-name>"` - SSE messages can also trigger HTTP callbacks using
  the [`hx-trigger`](https://htmx.org/attributes/hx-trigger) attribute.
* `sse-close=<message-name>` - To close the EventStream gracefully when that message is received. This might be helpful
  if you want to send information to a client that will eventually stop.



## Usage

```html

<div hx-ext="sse" sse-connect="/chatroom" sse-swap="message">
    Contents of this box will be updated in real time
    with every SSE message received from the chatroom.
</div>
```

### Connecting to an SSE Server

To connect to an SSE server, use the `hx-ext="sse"` attribute to install the extension on that HTML element, then
add `sse-connect="<url>"` to the element to make the connection.

When designing your server application, remember that SSE works just like any HTTP request. Although you cannot send any
messages to the server after you have established a connection, you can send parameters to the server along with your
request. So, instead of making an SSE connection to your server at `https://my-server/chat-updates` you can also connect
to `https://my-server/chat-updates?friends=true&format=detailed`. This allows your server to customize its responses to
what your client needs.

### Receiving Named Events

SSE messages consist of an event name and a data packet. No other metadata is allowed in the message. Here is an
example:

```txt
event: EventName
data: <div>Content to swap into your HTML page.</div>
```

We'll use the `sse-swap` attribute to listen for this event and swap its contents into our webpage.

```html

<div hx-ext="sse" sse-connect="/event-source" sse-swap="EventName"></div>
```

Notice that the name `EventName` from the server's message must match the value in the `sse-swap` attribute. Your server
can use as many different event names as necessary, but be careful: browsers can only listen for events that have been
explicitly named. So, if your server sends an event named `ChatroomUpdate` but your browser is only listening for events
named `ChatUpdate` then the extra event will be discarded.

### Receiving Unnamed Events

SSE messages can also be sent without any event name. In this case, the browser uses the default name `message` in its
place. The same rules specified above still apply. If your server sends an unnamed message, then you must listen for it
by including `sse-swap="message"`. There is no option for using a catch-all name. Here's how this looks:

```txt
data: <div>Content to swap into your HTML page.</div>
```

```html

<div hx-ext="sse" sse-connect="/event-source" sse-swap="message"></div>
```

### Receiving Multiple Events

You can also listen to multiple events (named or unnamed) from a single EventSource. Listeners must be either 1) the
same element that contains the `hx-ext` and `sse-connect` attributes, or 2) child elements of the element containing
the `hx-ext` and `sse-connect` attributes.

```html

Multiple events in the same element
<div hx-ext="sse" sse-connect="/server-url" sse-swap="event1,event2"></div>

Multiple events in different elements (from the same source).
<div hx-ext="sse" sse-connect="/server-url">
    <div sse-swap="event1"></div>
    <div sse-swap="event2"></div>
</div>
```

### Trigger Server Callbacks

When a connection for server sent events has been established, child elements can listen for these events by using the
special [`hx-trigger`](https://htmx.org/attributes/hx-trigger) syntax `sse:<event_name>`. This, when combined with
an `hx-get` or similar will trigger the element to make a request.

Here is an example:

```html

<div hx-ext="sse" sse-connect="/event_stream">
    <div hx-get="/chatroom" hx-trigger="sse:chatter">
        ...
    </div>
</div>
```

This example establishes an SSE connection to the `event_stream` end point which then triggers
a `GET` to the `/chatroom` url whenever the `chatter` event is seen.

### Automatic Reconnection

If the SSE Event Stream is closed unexpectedly, browsers are supposed to attempt to reconnect automatically. However, in
rare situations this does not work and your browser can be left hanging. This extension adds its own reconnection
logic (using an [exponential-backoff algorithm](https://en.wikipedia.org/wiki/Exponential_backoff)) on top of the
browser's automatic reconnection, so that your SSE streams will always be as reliable as possible.

### Testing SSE Connections with the Demo Server

Htmx includes a demo SSE server written in Node.js that will help you to see SSE in action, and begin bootstrapping your
own SSE code. It is located in the /test/ws-sse folder of
the [`htmx-extensions`](https://github.com/bigskysoftware/htmx-extensions) repository. Look at /test/ws-sse/README.md
for instructions on running and using the test server.

### Migrating from Previous Versions

Previous versions of htmx used a built-in tag `hx-sse` to implement Server Sent Events. This code has been migrated into
an extension instead. Here are the steps you need to take to migrate to this version:

| Old Attribute                  | New Attribute            | Comments                                                                                                                                                                                        |
|--------------------------------|--------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `hx-sse=""`                    | `hx-ext="sse"`           | Use the `hx-ext="sse"` attribute to install the SSE extension into any HTML element.                                                                                                            |
| `hx-sse="connect:<url>"`       | `sse-connect="<url>"`    | Add a new attribute `sse-connect` to the tag that specifies the URL of the Event Stream.  This attribute must be in the same tag as the `hx-ext` attribute.                                     |
| `hx-sse="swap:<EventName>"`    | `sse-swap="<EventName>"` | Add a new attribute `sse-swap` to any elements that will be swapped in via the SSE extension.  This attribute must be placed **on** or **inside of** the tag containing the `hx-ext` attribute. |
| `hx-trigger="sse:<EventName>"` | NO CHANGE                | any `hx-trigger` attributes do not need to change.  The extension will identify these attributes and add listeners for any events prefixed with `sse:`                                          |

### Listening to events dispatched by this extension

This extension dispatches several events. You can listen for these events like so:

```javascript
document.body.addEventListener('htmx:sseBeforeMessage', function (e) {
    // do something before the event data is swapped in
})
```

Each event object has a `detail` field that contains details of the event.

#### `htmx:sseOpen`

This event is dispatched when an SSE connection has been successfully established.

##### Details

* `detail.elt` - The element on which the SSE connection was setup. This is the element which has the `sse-connect`
  attribute.
* `detail.source` - The [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource) object.

#### `htmx:sseError`

This event is dispatched when an SSE connection could not be established.

##### Details

* `detail.error` - The error that occurred while creating
  an [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource).
* `detail.source` - The [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource).

#### `htmx:sseBeforeMessage`

This event is dispatched just before the SSE event data is swapped into the DOM. If you don't want to swap
call `preventDefault()` on the event. Additionally the `detail` field is
a [MessageEvent](https://developer.mozilla.org/en-US/docs/Web/API/EventSource/message_event) - this is the event created
by [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource) when it receives an SSE message.

##### Details

* `detail.elt` - The swap target.

#### `htmx:sseMessage`

This event is dispatched after the SSE event data has been swapped into the DOM. The `detail` field is
a [MessageEvent](https://developer.mozilla.org/en-US/docs/Web/API/EventSource/message_event) - this is the event created
by [EventSource](https://developer.mozilla.org/en-US/docs/Web/API/EventSource) when it receives an SSE message.

#### `htmx:sseClose`

This event is dispatched in three different closing scenario. To control for the scenario the user can control for the
evt.detail.sseclose property.

```javascript
document.body.addEventListener('htmx:sseClose', function (e) {
    const reason = e.detail.type
    switch (reason) {
        case "nodeMissing":
            // Parent node is missing and therefore connection was closed
        ...
        case "nodeReplaced":
            // Parent node replacement caused closing of connection
        ...
        case "message":
            // connection was closed due to reception of message sse-close
        ...
    }
})
```

## Server-Sent Events (SSE) in FastHTML with HTMX 2

Server-sent events (SSE) allow a server to push new data to a web page at any time. Unlike WebSockets, SSE is **unidirectional** (server to client only) and is part of the HTTP specification.

### Complete Example

FastHTML provides several tools for working with SSE. Here's a complete example with detailed annotations:

```python
import random
from asyncio import sleep
from fasthtml.common import *

# 1. Import the HTMX SSE extension
hdrs = (Script(src="https://unpkg.com/htmx-ext-sse@2.2.1/sse.js"),)
app, rt = fast_app(hdrs=hdrs)

@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate pairs of random numbers, as the list grows scroll downwards."),
        Div(
            hx_ext="sse",                    # 2. Tell HTMX to load the SSE extension
            sse_connect="/number-stream",    # 3. Look at /number-stream for SSE content
            hx_swap="beforeend show:bottom", # 4. Add new items at end and scroll down
            sse_swap="message"               # 5. Specify the event name
        )
    )

# 6. Set up the asyncio event loop
shutdown_event = signal_shutdown()

# 7. Don't forget to make this an async function!
async def number_generator():
    # 8. Iterate through the asyncio event loop
    while not shutdown_event.is_set():
        # 9. Yield data as FT components
        data = Article(random.randint(1, 100))
        yield sse_message(data)
        await sleep(1)

# 10. The endpoint must be async and return an EventStream
@rt("/number-stream")
async def get(): 
    return EventStream(number_generator())
```

### Key Points Explained

1. **Import the extension**: Add the HTMX SSE extension script to your app headers
2. **Load extension**: Use `hx_ext="sse"` on the container element
3. **Connect to endpoint**: `sse_connect` points to your SSE endpoint URL
4. **Configure swap**: `hx_swap="beforeend show:bottom"` adds items at the end and auto-scrolls
5. **Event name**: `sse_swap="message"` is FastHTML's default event name (only change if you have multiple SSE endpoints)
6. **Async setup**: Use `signal_shutdown()` for clean shutdown handling
7. **Async generator**: The generator function must be `async`
8. **Loop control**: Check `shutdown_event.is_set()` to handle graceful shutdown
9. **Yield FT components**: Data should be FastHTML components for seamless HTMX integration
10. **EventStream return**: The endpoint must return `EventStream(generator())`


## example of using SSE htmx 4 with fasthtml (old version, need to update)

```python
import json
from fasthtml.common import *
from fastcore.meta import use_kwargs, delegates
import asyncio
from claudette import Client as ClaudetteClient
from claudette import models
from datetime import datetime
from fasthtml.common import *
from starlette.responses import StreamingResponse
from fasthtml.jupyter import *
from asyncio import create_task

@delegates(ft_hx)
def Partial(*args, **kwargs):
    return ft_hx("hx-partial")(*args, **kwargs)
# srv.stop()
cli = ClaudetteClient(models[1])
sp = """You are a helpful and concise assistant."""
messages = []

# Set up the app, including daisyui and tailwind and the htmx sse extension for the chat component
tlink = (Script(src="https://cdn.tailwindcss.com"),)
dlink = Link(
    rel="stylesheet",
    href="https://cdn.jsdelivr.net/npm/daisyui@4.11.1/dist/full.min.css",
)

htmx_v4 = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha4/dist/htmx.js")

config = """{
    "metaCharacter":"-",
    "sse": {
        "reconnect": true,
        "reconnectMaxAttempts": 10,
        "reconnectDelay": 500,
        "reconnectMaxDelay": 60000,
        "reconnectJitter": 0.3,
        "pauseInBackground": false
    }
}
"""

# config = """{
#     "metaCharacter":"-"
# }
# """

meta_cfg = Meta(name="htmx-config", content=config)
app,rt = fast_app(live=True, htmx=False, hdrs=(tlink, dlink, picolink, meta_cfg, fhjsscr, htmx_v4))

def ChatMessage(msg_idx, streaming=False, **kwargs):
    msg = messages[msg_idx]
    bubble_class = "chat-bubble-primary" if msg["role"] == "user" else "chat-bubble-secondary"
    chat_class = "chat-end" if msg["role"] == "user" else "chat-start"
    stream_attrs = dict(hx_get=f"/get-message?msg_idx={msg_idx}", hx_swap="beforeend show:bottom", hx_trigger="load") if streaming else {}
    return Div(
        Div(msg["role"], cls="chat-header"),
        Div(msg["content"], id=f"chat-content-{msg_idx}", cls=f"chat-bubble {bubble_class}", **stream_attrs, **kwargs),
        id=f"chat-message-{msg_idx}",
        cls=f"chat {chat_class}")

def ChatInput():
    return Input(
        type="text",
        name="msg",
        id="msg-input",
        placeholder="Type a message",
        cls="input input-bordered w-full",
        hx_swap_oob="true",
    )
    
@app.route("/")
def get():
    page = Body(
        H1("Chatbot SSE (server-sent events) Demo"),
        Div(
            *[ChatMessage(i) for i in range(len(messages))],
            id="chatlist",
            cls="chat-box h-[73vh] overflow-y-auto",
        ),
        Form(
            Group(ChatInput(), Button("Send", cls="btn btn-primary")),
            hx_post="/send-message",
            hx_target="#chatlist",
            hx_swap="beforeend",
            cls="flex space-x-2 mt-2",
        ),
        cls="p-4 max-w-lg mx-auto",
    )
    return Title("Chatbot Demo"), page

active_streams = {}

async def fetch_response(msg_idx):
    r = cli(messages[:-1], sp=sp, stream=True)
    for chunk in r:
        messages[msg_idx]["content"] += chunk
        await asyncio.sleep(0.1)
    active_streams.pop(msg_idx, None)

@app.post("/send-message")
async def send_message(msg: str):
    messages.append({"role": "user", "content": msg})
    user_msg = Div(ChatMessage(len(messages) - 1))
    messages.append({"role": "assistant", "content": ""})
    msg_idx = len(messages) - 1
    active_streams[msg_idx] = True
    create_task(fetch_response(msg_idx))
    assistant_msg = Div(ChatMessage(msg_idx, streaming=True))
    return user_msg, assistant_msg, ChatInput()

async def message_generator(msg_idx, last_id=0):
    last_sent = last_id
    while msg_idx in active_streams or last_sent < len(messages[msg_idx]["content"]):
        content = messages[msg_idx]["content"]
        if len(content) > last_sent:
            new_content = content[last_sent:]
            last_sent = len(content)
            yield f"id: {last_sent}\ndata: {new_content}\n\n"
        await asyncio.sleep(0.1)
    final_el = Partial(Div(messages[msg_idx]["content"], cls="chat-bubble chat-bubble-secondary"), hx_target=f"#chat-content-{msg_idx}", hx_swap="outerHTML")
    data_lines = '\n'.join(f'data: {line}' for line in to_xml(final_el).splitlines())
    yield f"{data_lines}\n\n"

@app.get("/get-message")
async def get_message(request, msg_idx: int):
    last_id = int(request.headers.get("Last-Event-ID", 0))
    return StreamingResponse(message_generator(msg_idx, last_id), media_type="text/event-stream")
    
srv = JupyUvi(app)
```

# Full documents

In [ ]:
# import httpx, os
# 
# base_url = "https://raw.githubusercontent.com/bigskysoftware/htmx/four/www/src/content/docs"
# out_dir = "/app/data/fasthtml-example/htmx4-docs"
# 
# files = {
#     "01-get-started": ["01-installation.md", "02-migration.md", "index.md"],
#     "02-core-concepts": ["01-mental-model.md", "02-hypermedia-controls.md", "03-requests-and-responses.md", "04-client-scripting.md", "05-multi-target-updates.md", "index.md"],
#     "03-features": ["01-css-transitions.md", "02-synchronization.md", "03-confirmations.md", "04-boosting.md", "05-history.md", "06-validation.md", "07-web-components.md", "08-attribute-inheritance.md", "09-extended-selectors.md", "index.md"],
#     "06-extensions": ["01-using-extensions.md", "02-extension-migration.md", "03-sse.md", "04-ws.md", "05-head-support.md", "06-preload.md", "07-htmx-2-compat.md", "08-optimistic.md", "09-download.md", "10-upsert.md", "11-targets.md", "12-ptag.md", "13-browser-indicator.md", "14-alpine-compat.md", "index.md"],
#     "07-security": ["01-best-practices.md", "02-caching.md", "index.md"],
#     "08-troubleshoot": ["01-debugging.md", "02-configuration.md", "index.md"],
# }
# 
# client = httpx.Client(follow_redirects=True)
# count = 0
# for folder, filenames in files.items():
#     folder_path = f"{out_dir}/{folder}"
#     os.makedirs(folder_path, exist_ok=True)
#     for fn in filenames:
#         url = f"{base_url}/{folder}/{fn}"
#         resp = client.get(url)
#         if resp.status_code == 200:
#             with open(f"{folder_path}/{fn}", "w") as f:
#                 f.write(resp.text)
#             count += 1
#         else:
#             print(f"FAILED ({resp.status_code}): {url}")
# client.close()
# 
# # Also grab the root index
# resp = httpx.get(f"{base_url}/index.mdx", follow_redirects=True)
# if resp.status_code == 200:
#     with open(f"{out_dir}/index.mdx", "w") as f:
#         f.write(resp.text)
#     count += 1
# 
# print(f"Downloaded {count} files to {out_dir}")

Downloaded 41 files to /app/data/fasthtml-example/htmx4-docs


All 41 htmx v4 doc files have been downloaded to:

**`/app/data/fasthtml-example/htmx4-docs/`**

Here's the structure and what each section covers:

| Folder | Key Files | Summary |
|---|---|---|
| `01-get-started/` | installation, **migration**, index | How to install htmx 4 and migrate from v2 (migration guide is already in our dialog notes above) |
| `02-core-concepts/` | mental-model, hypermedia-controls, requests-and-responses, client-scripting, multi-target-updates | How htmx thinks about HTML-driven apps, making requests, scripting integration, and updating multiple DOM targets (`hx-partial`, `hx-swap-oob`) |
| `03-features/` | css-transitions, synchronization, confirmations, boosting, history, validation, web-components, **attribute-inheritance**, extended-selectors | Detailed feature docs — **attribute-inheritance** is especially important since v4 switched to explicit inheritance |
| `06-extensions/` | using-extensions, extension-migration, **sse**, **ws**, head-support, preload, htmx-2-compat, optimistic, download, upsert, targets, ptag, browser-indicator, alpine-compat | All extension docs — **SSE and WS** are key for our FastHTML patching work; **htmx-2-compat** documents the backward-compat shim |
| `07-security/` | best-practices, caching | Security considerations and caching guidance |
| `08-troubleshoot/` | debugging, configuration | How to debug htmx and all configuration options |

use `view` to read , e.g. `view("/app/data/fasthtml-example/htmx4-docs/03-features/08-attribute-inheritance.md")` whenever we need to look something up.

# solvecdp


<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

fastcdp provides an async Python client for the [Chrome DevTools
Protocol](https://chromedevtools.github.io/devtools-protocol/) (CDP)
over solveit’s js bridge. It exposes every CDP domain as a Python
attribute with auto-generated signatures and docstrings —
e.g. `await cdp.page.navigate(url=...)`.

It includes event subscription via `cdp.on()`/`cdp.wait_event()`,
navigation helpers (`goto`, `wait_for_selector`, `wait_for`), screenshot
capture, and accessibility tree access. A
[`cdp_search`](https://AnswerDotAI.github.io/solvecdp/core.html#cdp_search)
utility lets you search CDP commands by name or description. For use
inside [safepyrun](https://github.com/AnswerDotAI/safepyrun) sandboxes,
[`cdp_yolo()`](https://AnswerDotAI.github.io/solvecdp/core.html#cdp_yolo)
registers all CDP classes.

## Installation

You must install the [solveit-chrome
extension](https://github.com/AnswerDotAI/solveit-chrome) before using
solvecdp.

Solveit already has solvecdp installed, but if you want to install the
latest you can get it from [pypi](https://pypi.org/project/solvecdp/):

``` sh
$ pip install solvecdp
```

## How to use

``` python
from solvecdp import *
```

### The JsCDP class

Every CDP domain is available as an attribute with auto-generated
signatures. You can search for commands with
[`cdp_search`](https://AnswerDotAI.github.io/solvecdp/core.html#cdp_search):

``` python
cdp_search('screenshot')
```

    "Emulation.setVisibleSize: Resizes the frame/viewport of the page. Note that this does not affect the frame's container\n(e.g. browser window). Can \nHeadlessExperimental.beginFrame: Sends a BeginFrame to the target and returns when the frame was completed. Optionally captures a\nscreenshot from the res\n  evt Overlay.screenshotRequested: Fired when user asks to capture screenshot of some area on the page.\nPage.captureScreenshot: Capture page screenshot."

Create a new page:

``` python
jc = await JsCDP.new()
```

Go to a page:

``` python
await jc.goto('https://httpbin.org/forms/post')
```

Eval js:

``` python
await jc.eval('document.title')
```

    '6. httpbin.org/forms/post'

Or you can `wait_for` any js expression to be truthy, and have it
returned:

``` python
await jc.wait_for('document.title')
```

    '6. httpbin.org/forms/post'

Take a screenshot of the page:

``` python
img = await jc.screenshot()
```

Clean up when done:

``` python
await jc.close()
```

See [`JsCDP`](https://AnswerDotAI.github.io/solvecdp/core.html#jscdp)
docs for full details.

## Filling forms

``` python
page = await JsCDP.new(url='https://httpbin.org/forms/post')
```

For finding elements to interact with, use `ax_tree`:

``` python
root = await page.ax_tree()
print(str(root)[:300])
```

    - **RootWebArea** "6. httpbin.org/forms/post" `focusable=True` `url=https://httpbin.org/forms/post` [#14]
      - **LabelText** "" [#20]
        - **StaticText** "Customer name: " [#62]
          - **InlineTextBox** "Customer name: "
        - **textbox** "Customer name: " `focusable=True` `editable=plaintext` `set

`find` and `find_id` are used to identify elements in the tree:

``` python
nmid = root.find_id('textbox', 'Customer name')
nmid
```

    2

You can use regular CDP methods, or one of the provided shortcuts:

``` python
await page.fill_text(nmid, 'Jeremy Howard')
await page.click(root.find_id('radio', 'Large'))
await page.js_node_run('this.value = "18:30"', root.find_id('InputTime', 'delivery time'));
```

You can use `click` to click a button, or `click_and_wait` to wait for
the next page to load:

``` python
await page.click_and_wait(root.find_id('button', 'Submit order'))
```

``` python
await page.close()
```

To allow LLMs like solveit with safepyrun to access solvecdp, use:

``` python
cdp_yolo()
```

Then use a prompt such as:

> Try using pyrun to create a `page_` JsCDP object, then goto `<url>`,
> fill it out, read it to check it’s filled correctly, then submit it,
> and see what you get back. Don’t use find_id - you can get all the ids
> at once with ax_tree (don’t truncate the result of it). Don’t add
> extra waits etc - solvecdp handles it automatically. IDs can change so
> be sure to use the ax_tree IDs you read.